# Research Assistant Agent

Follow two questions through the same LangGraph workflow used by the CLI and API. The default run uses deterministic providers, so it runs offline and needs no API keys.

Run from the repository root using the Python environment where `requirements.txt` is installed.

## Research turn graph

![Research turn graph with parallel local and web retrieval, a coverage retry, and the cited response](graph.png)

The session graph keeps conversation history and the latest result in an in-memory checkpoint. Each question runs in a stateless subgraph. Query plans, search candidates, and retry counters stay within that turn.

## Choose providers

Leave `USE_LIVE_PROVIDERS` off for the reproducible demo. The offline query adapter uses prior conversation to expand the follow-up, and the offline response displays the top selected passage with its citation.

Set `USE_LIVE_PROVIDERS = True` to run Gemini rewriting, embeddings, reranking, and answer generation. Add `GOOGLE_API_KEY` to `.env` and optionally add `TAVILY_API_KEY` for web evidence. Live traces include the text of the prepared queries.

In [1]:
import json
import os
import subprocess
import sys
from dataclasses import replace
from uuid import uuid4

from dotenv import load_dotenv

from evaluation.retrieval_benchmark import _offline_providers
from graph import build_graph

USE_LIVE_PROVIDERS = False


def demo_answer(_question, snippets, _gaps):
    return f"Top-ranked passage [1]\n\n{snippets[0]['content']}"


if USE_LIVE_PROVIDERS:
    load_dotenv()
    if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
        raise RuntimeError("Set GOOGLE_API_KEY before enabling live providers.")
    providers = None
else:
    providers = replace(_offline_providers(), answer=demo_answer)

session_graph = build_graph(providers)
config = {"configurable": {"thread_id": str(uuid4())}}
print("Provider mode", "live" if USE_LIVE_PROVIDERS else "offline simulation")

Provider mode offline simulation


## Stream a research turn

Watch the query arrive before either search branch. Fusion combines the two branches, and reranking selects the evidence that reaches the response. If the coverage check finds gaps, the graph can make one more retrieval pass.

In [2]:
def show_turn(topic):
    print(f"Question  {topic}\n")
    for namespace, event in session_graph.stream(
        {"topic": topic}, config, stream_mode="updates", subgraphs=True
    ):
        if not namespace:
            continue
        for stage, update in event.items():
            if stage == "prepare_queries":
                print(f"{stage}  round {update['retries'] + 1}")
                for query in update["queries"]:
                    print(f"  - {query}")
            elif stage == "retrieve_local":
                print(f"{stage}  {len(update['local_rankings'])} accumulated rankings")
            elif stage == "retrieve_web":
                print(f"{stage}  {sum(map(len, update['web_rankings']))} accumulated hits")
            elif stage == "fuse_candidates":
                print(f"{stage}  {len(update['fused_candidates'])} unique candidates")
            elif stage == "rerank_candidates":
                titles = ", ".join(item["title"] for item in update["selected_candidates"])
                print(f"{stage}  {titles or 'no evidence selected'}")
            elif stage == "check_coverage":
                print(f"{stage}  {update['coverage']}")
            else:
                print(stage)

    result = session_graph.get_state(config).values
    print(f"\n{result['final_answer']}")
    if result["sources"]:
        print("\nSources")
        for number, source in enumerate(result["sources"], 1):
            print(f"[{number}] {source}")
    return result


first_turn = show_turn("How does retrieval-augmented generation work?")

Question  How does retrieval-augmented generation work?

prepare_queries  round 1
  - How does retrieval-augmented generation work?
retrieve_local  2 accumulated rankings
retrieve_web  0 accumulated hits
fuse_candidates  6 unique candidates
rerank_candidates  Retrieval-Augmented Generation (RAG)
check_coverage  sufficient
respond

Top-ranked passage [1]

RAG combines a retrieval component with a generative model. At inference time, relevant documents are fetched from a knowledge base and prepended to the prompt. This reduces hallucination and allows the model to cite sources. Common retrieval backends include FAISS, Chroma, Pinecone, and Weaviate. Dense passage retrieval (DPR) and BM25 are popular scoring methods.

Sources
[1] internal://docs/rag_overview


## Ask a follow-up

The same thread ID lets the query adapter use the previous exchange when it prepares a search for “it.” The checkpoint field list shows what the session retains after the turn.

In [3]:
follow_up = show_turn("How does it reduce hallucinations?")
print("\nCheckpoint fields", ", ".join(sorted(follow_up)))
print("Conversation messages", len(follow_up["messages"]))

Question  How does it reduce hallucinations?

prepare_queries  round 1
  - How does retrieval-augmented generation work? How does it reduce hallucinations?
retrieve_local  2 accumulated rankings
retrieve_web  0 accumulated hits
fuse_candidates  6 unique candidates
rerank_candidates  Retrieval-Augmented Generation (RAG)
check_coverage  sufficient
respond

Top-ranked passage [1]

RAG combines a retrieval component with a generative model. At inference time, relevant documents are fetched from a knowledge base and prepended to the prompt. This reduces hallucination and allows the model to cite sources. Common retrieval backends include FAISS, Chroma, Pinecone, and Weaviate. Dense passage retrieval (DPR) and BM25 are popular scoring methods.

Sources
[1] internal://docs/rag_overview

Checkpoint fields final_answer, messages, sources, topic
Conversation messages 4


## Inspect retrieval quality

This cell runs the 24-case offline simulation. It shows how the evaluator compares Recall@5, nDCG@5, latency, and out-of-corpus behavior. These scores describe the deterministic fixtures. The credentialed commands and recorded live results are in the [README](README.md#verify-retrieval).

In [4]:
report = json.loads(subprocess.check_output(
    [sys.executable, "-m", "evaluation.retrieval_benchmark"], text=True
))
print(f"{report['mode']}  {report['cases']} cases, {report['scored_cases']} scored")
for name, metrics in report["metrics"].items():
    print(
        f"{name}  Recall@5 {metrics['recall_at_5']:.3f}, "
        f"nDCG@5 {metrics['ndcg_at_5']:.3f}, mean {metrics['mean_ms']:.2f} ms, "
        f"out-of-corpus hits {metrics['out_of_corpus_with_results']}/{metrics['out_of_corpus_total']}"
    )

offline_simulation  24 cases, 21 scored
lexical  Recall@5 0.619, nDCG@5 0.601, mean 0.36 ms, out-of-corpus hits 1/3
hybrid_before_rerank  Recall@5 0.857, nDCG@5 0.794, mean 5.52 ms, out-of-corpus hits 3/3
full_retrieval  Recall@5 0.619, nDCG@5 0.618, mean 6.76 ms, out-of-corpus hits 1/3
